# 🐦 Tweet Sentiment Classifier — End-to-End Workshop

**What you'll build:** a full pipeline that takes raw tweets and predicts sentiment
(Negative / Neutral / Positive) — starting with classical ML baselines, moving up to
a Bidirectional LSTM deep learning model, and finishing with a deployable
**Streamlit** app.

### Workflow

1. Dataset Overview — understand `Twitter_Data.csv`
2. NLP Preprocessing — `clean_text()` and label encoding
3. Train/Test Split — stratified sampling
4. Classical ML Baselines — Naive Bayes, Logistic Regression, Linear SVM (TF-IDF)
5. Tokenization & Padding — Keras `Tokenizer` and `pad_sequences`
6. Embeddings — trainable vs. pretrained (GloVe)
7. Deep Learning Model — BiLSTM sentiment classifier
8. Model Evaluation — metrics, confusion matrices, model comparison
9. Model Saving & Deployment — `sentiment_model.h5` + a Streamlit app

> 💡 **Why compare classical ML with deep learning?** In industry you rarely jump
> straight to a neural network. A fast, interpretable baseline (Naive Bayes / Logistic
> Regression on TF-IDF) tells you how hard the problem is and gives you a number to beat.
> If the BiLSTM can't clear that bar by a healthy margin, it isn't earning its extra
> training time and deployment complexity.

---
### 0. Environment Setup


In [ ]:
# Core
import re
import string
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Classical ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
print("TensorFlow:", tf.__version__)


## 1. Dataset Overview

We're using `Twitter_Data.csv`, which has two columns:

| column | meaning |
|---|---|
| `clean_text` | the raw tweet text (the name is a bit misleading — it still needs cleaning!) |
| `category` | sentiment label: `-1` = Negative, `0` = Neutral, `1` = Positive |

Place `Twitter_Data.csv` in the same folder as this notebook before running the cell below.

> 🎯 **Try it yourself:** before writing any code, always ask — *how many classes are
> there, are they balanced, how long are the texts, are there duplicates or nulls?*
> Every modelling decision downstream (loss function, class weighting, maxlen) traces
> back to what you find here.


In [ ]:
DATA_PATH = "Twitter_Data.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
# Basic health checks
print("Nulls per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Drop rows with missing text/label — a handful of NaNs is common in this dataset
df = df.dropna(subset=["clean_text", "category"]).reset_index(drop=True)
df["category"] = df["category"].astype(int)
print("Shape after cleanup:", df.shape)


In [ ]:
# Class balance
label_map = {-1: "Negative", 0: "Neutral", 1: "Positive"}
df["sentiment"] = df["category"].map(label_map)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="sentiment", order=["Negative", "Neutral", "Positive"],
              palette=["#e74c3c", "#95a5a6", "#2ecc71"], ax=ax)
ax.set_title("Class distribution")
plt.show()

df["sentiment"].value_counts(normalize=True).round(3)


In [ ]:
# Tweet length distribution — informs our `maxlen` choice for padding later
df["word_count"] = df["clean_text"].astype(str).apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df["word_count"], bins=40, ax=ax)
ax.axvline(df["word_count"].quantile(0.95), color="red", linestyle="--",
           label="95th percentile")
ax.set_title("Tweet length (words)")
ax.legend()
plt.show()

print(df["word_count"].describe())


> **Note on imbalance:** this dataset usually skews Positive-heavy. We'll keep that
> in mind — it's why we use **stratified** splitting (Section 3) and why accuracy
> alone won't tell the full story (Section 8, macro-F1 matters more).

## 2. NLP Preprocessing — `clean_text()` and Label Encoding

Raw tweets are messy: URLs, `@mentions`, `#hashtags`, emojis, retweet markers,
inconsistent casing, elongated words ("sooooo good"), and slang.

**Best-practice principle:** *clean only what hurts your model, keep what helps it.*

- For **TF-IDF / classical ML**, aggressive cleaning (lowercasing, stripping
  punctuation/numbers, removing stopwords) helps — sparse bag-of-words models have
  no way to "learn past" noise.
- For **deep learning with embeddings**, be gentler. Stopwords and word order carry
  signal that an LSTM can exploit; over-cleaning can actually hurt it. We'll keep a
  lighter-touch version of the text for the BiLSTM.

So we'll build **two levels of cleaning** from one base function.


In [ ]:
import contractions  # pip install contractions  (expands "don't" -> "do not")

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_SYMBOL_RE = re.compile(r"#")
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s]")
MULTI_SPACE_RE = re.compile(r"\s+")
ELONGATED_RE = re.compile(r"(.)\1{2,}")  # "sooooo" -> "soo"

def base_clean(text: str) -> str:
    '''Shared first pass: lowercase, strip URLs/mentions, expand contractions.'''
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_SYMBOL_RE.sub("", text)       # keep the word, drop the '#'
    text = contractions.fix(text)
    text = ELONGATED_RE.sub(r"\1\1", text)       # normalize elongation
    text = NON_ALPHA_RE.sub(" ", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text


from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text_classical(text: str) -> str:
    '''Aggressive cleaning for TF-IDF / classical ML models.'''
    text = base_clean(text)
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in STOPWORDS and len(w) > 1]
    return " ".join(tokens)

def clean_text_dl(text: str) -> str:
    '''Lighter cleaning for the BiLSTM — keep stopwords & word order.'''
    return base_clean(text)


In [ ]:
df["text_classical"] = df["clean_text"].apply(clean_text_classical)
df["text_dl"] = df["clean_text"].apply(clean_text_dl)

df[["clean_text", "text_classical", "text_dl"]].sample(5, random_state=SEED)


In [ ]:
# Label encoding — keep the mapping, we'll need to invert it at inference time
le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])   # -1,0,1 -> 0,1,2 (alphabetical/sorted order)

print("Classes:", le.classes_)
print("Encoded mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Save this mapping now — you WILL need it for inference later
LABEL_MAP_INV = {int(code): label_map[int(orig)] for code, orig in zip(le.transform(le.classes_), le.classes_)}
print("Inference-time mapping:", LABEL_MAP_INV)


> 🎯 **Try it yourself:** print 10 random rows where `text_classical` ends up empty
> after cleaning (very short tweets, all-stopword tweets). Decide: drop them, or keep
> and let the model predict from an empty/near-empty input? There's no single right
> answer — document your choice.

## 3. Train/Test Split — Stratified Sampling

With an imbalanced dataset, a *random* split can accidentally starve the minority
class in the test set. **Stratified** splitting preserves the class proportions in
both train and test sets.


In [ ]:
X_text_classical = df["text_classical"].values
X_text_dl = df["text_dl"].values
y = df["label"].values

(X_train_classical, X_test_classical,
 X_train_dl, X_test_dl,
 y_train, y_test) = train_test_split(
    X_text_classical, X_text_dl, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,          # <-- this is the key line
)

print("Train:", X_train_classical.shape[0], " Test:", X_test_classical.shape[0])
print("Train class balance:\n", pd.Series(y_train).value_counts(normalize=True).round(3))
print("Test class balance:\n", pd.Series(y_test).value_counts(normalize=True).round(3))


## 4. Classical ML Baselines (TF-IDF)

Three fast, strong baselines:

- **Multinomial Naive Bayes** — the classic text-classification baseline. Assumes
  word independence given the class; surprisingly hard to beat on short text.
- **Logistic Regression** — a linear model with well-calibrated probabilities,
  usually the strongest of the classical baselines.
- **Linear SVM** — often edges out logistic regression on high-dimensional sparse
  TF-IDF features, at the cost of well-calibrated probabilities.

We fit **one shared TF-IDF vectorizer** on the training data only (never fit on
test data — that's a form of leakage) and reuse it for all three models.


In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),   # unigrams + bigrams capture short phrases like "not good"
    min_df=2,
)

X_train_tfidf = tfidf.fit_transform(X_train_classical)
X_test_tfidf = tfidf.transform(X_test_classical)

print("TF-IDF matrix shape:", X_train_tfidf.shape)


In [ ]:
classical_models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Linear SVM": LinearSVC(class_weight="balanced"),
}

classical_results = {}

for name, model in classical_models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    classical_results[name] = {"model": model, "preds": preds, "accuracy": acc, "macro_f1": f1}
    print(f"{name:22s} | accuracy={acc:.4f} | macro-F1={f1:.4f}")


In [ ]:
# Closer look at the strongest classical baseline
best_classical_name = max(classical_results, key=lambda k: classical_results[k]["macro_f1"])
best_classical = classical_results[best_classical_name]
print(f"Best classical baseline: {best_classical_name}\n")
print(classification_report(y_test, best_classical["preds"],
                             target_names=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)]))

cm = confusion_matrix(y_test, best_classical["preds"])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)],
            yticklabels=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_classical_name}")
plt.show()


> 🎯 **Try it yourself:** swap `TfidfVectorizer` for `CountVectorizer` and re-run
> Naive Bayes. Does accuracy change much? (Hint: Naive Bayes' independence
> assumption pairs naturally with raw counts — TF-IDF weighting sometimes helps
> less here than it does for Logistic Regression / SVM.)

## 5. Tokenization & Padding — Keras `Tokenizer` and `pad_sequences`

Neural networks need fixed-size numeric input. The pipeline is:

1. **Tokenizer** — builds a word→integer vocabulary from the training data.
2. **texts_to_sequences** — converts each tweet into a list of integers.
3. **pad_sequences** — pads/truncates every sequence to the same length (`MAXLEN`).

We fit the tokenizer on `text_dl` (the lightly-cleaned version) since we want the
embedding layer to see stopwords and natural word order.


In [ ]:
VOCAB_SIZE = 20000
MAXLEN = int(df["word_count"].quantile(0.95))   # covers 95% of tweets without truncation
OOV_TOKEN = "<OOV>"

print("Chosen MAXLEN:", MAXLEN)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_dl)   # fit ONLY on train — same leakage rule as TF-IDF

X_train_seq = tokenizer.texts_to_sequences(X_train_dl)
X_test_seq = tokenizer.texts_to_sequences(X_test_dl)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAXLEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAXLEN, padding="post", truncating="post")

print("Padded shape:", X_train_pad.shape)
print("Actual vocab size found:", len(tokenizer.word_index))

# One-hot targets for the softmax output layer
num_classes = len(np.unique(y))
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)


**Why `padding="post"` and `truncating="post"`?** For LSTMs, padding at the *end*
means the real content sits at the start of the sequence, close to the initial
hidden state — this is generally friendlier for recurrent layers than pre-padding
(though both are used in practice; try `"pre"` yourself and compare).

## 6. Embeddings — Trainable vs. Pretrained (GloVe)

An embedding layer maps each integer token to a dense vector. Two options:

| Approach | Pros | Cons |
|---|---|---|
| **Trainable embedding** (learned from scratch) | Simple, no extra downloads, adapts fully to this dataset's vocabulary/slang | Needs enough data to learn good vectors; struggles with rare words |
| **Pretrained (GloVe Twitter)** | Brings in general language knowledge, helps with rare words, usually faster to converge | Extra download; vocabulary mismatch for slang/misspellings not in GloVe |

We'll build the BiLSTM with a **trainable embedding** by default (simplest, no
external downloads required for the workshop), and include optional code to swap
in GloVe if you want to compare.


In [ ]:
EMBEDDING_DIM = 100

# --- Option A: trainable embedding (used below) ---
# handled directly inside the model in Section 7 via the Embedding layer.

# --- Option B (optional): pretrained GloVe Twitter vectors ---
# 1. Download glove.twitter.27B.100d.txt from https://nlp.stanford.edu/projects/glove/
# 2. Uncomment and run the block below to build an embedding matrix.

USE_GLOVE = False   # flip to True after downloading GloVe

def build_glove_embedding_matrix(glove_path, tokenizer, vocab_size, embedding_dim):
    embeddings_index = {}
    with open(glove_path, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = vector

    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    hits, misses = 0, 0
    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec
            hits += 1
        else:
            misses += 1
    print(f"GloVe coverage: {hits} hits, {misses} misses")
    return embedding_matrix

if USE_GLOVE:
    embedding_matrix = build_glove_embedding_matrix(
        "glove.twitter.27B.100d.txt", tokenizer, VOCAB_SIZE, EMBEDDING_DIM
    )


> 🎯 **Try it yourself:** if you have GloVe downloaded, set `USE_GLOVE = True`,
> pass `weights=[embedding_matrix], trainable=False` to the `Embedding` layer in
> Section 7, and compare validation accuracy/convergence speed against the
> trainable-from-scratch version.

## 7. Deep Learning Model — BiLSTM Sentiment Classifier

**Why Bidirectional?** A regular LSTM only reads left-to-right, so at any word it
only knows what came *before* it. A tweet like *"not good at all"* needs context
from *both* directions to correctly weight "not" against "good". A `Bidirectional`
wrapper runs two LSTMs — one forward, one backward — and concatenates their outputs.

**Architecture:**

```
Embedding  ->  SpatialDropout1D  ->  Bidirectional(LSTM)  ->  Dense(softmax)
```

- `SpatialDropout1D` drops entire embedding *dimensions* (not random individual
  values) — better suited to sequence data than plain `Dropout` right after an
  embedding layer.


In [ ]:
def build_bilstm_model(vocab_size, embedding_dim, maxlen, num_classes,
                        embedding_matrix=None, trainable_embedding=True):
    model = Sequential()
    if embedding_matrix is not None:
        model.add(Embedding(vocab_size, embedding_dim, input_length=maxlen,
                             weights=[embedding_matrix], trainable=trainable_embedding))
    else:
        model.add(Embedding(vocab_size, embedding_dim, input_length=maxlen))

    model.add(SpatialDropout1D(0.3))
    model.add(Bidirectional(LSTM(64, return_sequences=True)))
    model.add(Bidirectional(LSTM(32)))
    model.add(Dense(32, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model

bilstm_model = build_bilstm_model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    maxlen=MAXLEN,
    num_classes=num_classes,
)
bilstm_model.summary()


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint("sentiment_model.h5", monitor="val_loss", save_best_only=True),
]

history = bilstm_model.fit(
    X_train_pad, y_train_cat,
    validation_split=0.1,
    epochs=15,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()

plt.tight_layout()
plt.show()


> 🎯 **Try it yourself:** if train accuracy keeps climbing while val accuracy
> plateaus or drops, that's overfitting. Try increasing `SpatialDropout1D`'s rate,
> reducing LSTM units, or adding `recurrent_dropout` to the `LSTM` layers.

## 8. Model Evaluation — Metrics & Confusion Matrix

We already have per-model results for the classical baselines. Now evaluate the
BiLSTM the same way, then compare **everything** side by side.


In [ ]:
bilstm_probs = bilstm_model.predict(X_test_pad)
bilstm_preds = np.argmax(bilstm_probs, axis=1)

bilstm_acc = accuracy_score(y_test, bilstm_preds)
bilstm_f1 = f1_score(y_test, bilstm_preds, average="macro")

print(f"BiLSTM | accuracy={bilstm_acc:.4f} | macro-F1={bilstm_f1:.4f}\n")
print(classification_report(y_test, bilstm_preds,
                             target_names=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)]))

cm = confusion_matrix(y_test, bilstm_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)],
            yticklabels=[LABEL_MAP_INV[i] for i in sorted(LABEL_MAP_INV)], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — BiLSTM")
plt.show()


In [ ]:
# Head-to-head comparison of every model built in this notebook
comparison = pd.DataFrame({
    "Model": list(classical_results.keys()) + ["BiLSTM"],
    "Accuracy": [classical_results[m]["accuracy"] for m in classical_results] + [bilstm_acc],
    "Macro F1": [classical_results[m]["macro_f1"] for m in classical_results] + [bilstm_f1],
}).sort_values("Macro F1", ascending=False).reset_index(drop=True)

display(comparison)

fig, ax = plt.subplots(figsize=(7, 4))
comparison.set_index("Model")[["Accuracy", "Macro F1"]].plot(kind="bar", ax=ax)
ax.set_title("Model comparison")
ax.set_ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Reading this table:** if the BiLSTM's macro-F1 isn't meaningfully ahead of your
best classical baseline, that's a real, useful finding — not a failure. It tells you
the extra training time/complexity of the deep model may not be worth it for this
dataset size, and a TF-IDF + Logistic Regression pipeline might be the better
production choice. We're deploying the BiLSTM in this workshop for the *learning
value* of shipping a deep-learning model end-to-end — in a real project, let the
metrics in this table make that call.

## 9. Model Saving & Deployment Prep

To deploy, we need to persist **three** artifacts — not just the model:

1. `sentiment_model.h5` — the trained BiLSTM weights + architecture
2. `tokenizer.pickle` — so we can convert new text to the *same* integer sequence
3. `label_encoder.pickle` — so we can map predicted class indices back to names

Forgetting #2 or #3 is the single most common reason a "working" model breaks in
production — the app ends up building a *new* tokenizer at inference time, whose
word→integer mapping doesn't match what the model was trained on.


In [ ]:
# 1. Model (already saved by ModelCheckpoint above, but save explicitly too)
bilstm_model.save("sentiment_model.h5")

# 2. Tokenizer
with open("tokenizer.pickle", "wb") as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

# 3. Label encoder mapping (index -> human-readable sentiment)
with open("label_encoder.pickle", "wb") as f:
    pickle.dump(LABEL_MAP_INV, f, protocol=pickle.HIGHEST_PROTOCOL)

# 4. Config needed to reproduce preprocessing exactly at inference time
config = {"maxlen": MAXLEN, "vocab_size": VOCAB_SIZE}
with open("config.pickle", "wb") as f:
    pickle.dump(config, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved: sentiment_model.h5, tokenizer.pickle, label_encoder.pickle, config.pickle")


### Loading everything back and running inference

This is exactly the logic the Streamlit app will use — proving it here, in the
notebook, means we debug it once in a fast feedback loop instead of inside the app.


In [ ]:
loaded_model = load_model("sentiment_model.h5")

with open("tokenizer.pickle", "rb") as f:
    loaded_tokenizer = pickle.load(f)

with open("label_encoder.pickle", "rb") as f:
    loaded_label_map = pickle.load(f)

with open("config.pickle", "rb") as f:
    loaded_config = pickle.load(f)


def predict_sentiment(raw_text: str) -> dict:
    cleaned = clean_text_dl(raw_text)
    seq = loaded_tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=loaded_config["maxlen"], padding="post", truncating="post")
    probs = loaded_model.predict(padded, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return {
        "text": raw_text,
        "cleaned": cleaned,
        "sentiment": loaded_label_map[pred_idx],
        "confidence": float(probs[pred_idx]),
        "probabilities": {loaded_label_map[i]: float(p) for i, p in enumerate(probs)},
    }


for sample in [
    "I absolutely love how this turned out, best day ever!",
    "This is the worst service I have ever experienced.",
    "The event starts at 6pm tomorrow.",
]:
    result = predict_sentiment(sample)
    print(f"{result['sentiment']:9s} ({result['confidence']:.2%})  ->  {sample}")


> 🎯 **Try it yourself:** feed the model a deliberately tricky example —
> sarcasm ("oh great, another Monday") or mixed sentiment ("the food was amazing but
> the service was terrible"). BiLSTMs (and even humans!) often struggle with
> sarcasm since it depends on tone/context the text alone doesn't always carry.

## 10. Streamlit App

The full app lives in **`app.py`** next to this notebook (Streamlit apps run as
standalone scripts, not inside notebooks). It reuses the *exact* saved artifacts
from Section 9, so predictions match what you saw above exactly.

To run it:

```bash
pip install streamlit tensorflow
streamlit run app.py
```

The app:
- Loads `sentiment_model.h5`, `tokenizer.pickle`, `label_encoder.pickle`, `config.pickle` once (cached)
- Takes a tweet as free-text input
- Applies the **same** `clean_text_dl` preprocessing used in training
- Shows the predicted sentiment, confidence, and a probability breakdown chart

See `app.py` in this folder for the full source.


---
## Recap

| Stage | What we did | Why it matters |
|---|---|---|
| Preprocessing | Two cleaning levels: aggressive for TF-IDF, light for embeddings | Different model families need different amounts of noise removed |
| Split | Stratified train/test | Preserves class balance under imbalance |
| Classical ML | Naive Bayes, Logistic Regression, Linear SVM on TF-IDF | Fast, interpretable baseline to beat |
| Deep Learning | BiLSTM over a trainable/pretrained embedding | Captures word order & bidirectional context |
| Evaluation | Accuracy **and** macro-F1, confusion matrices, side-by-side comparison | Accuracy alone hides class imbalance problems |
| Deployment | Model + tokenizer + label map, all versioned together | The #1 real-world bug source is a mismatched tokenizer/model pair |

**Next steps to extend this workshop:** try a 1D-CNN or a small Transformer
encoder as a fourth model family, add k-fold cross-validation to the classical
baselines, or fine-tune a pretrained model like `distilbert-base-uncased` and add
it to the comparison table in Section 8.
